In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.18 Spin, Magnetic Moments, and the Electron in a Magnetic Field

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.18",
    title="Spin, Magnetic Moments, and the Electron in a Magnetic Field",
    blurb="The electron's hidden turn, made physical. We already know spin as the "
    "smallest angular momentum; here it acquires a body — a magnetic moment twice as "
    "strong as motion would give, an energy in a magnetic field that splits and "
    "precesses the spin, and a fourth quantum number that completes the atom and "
    "fixes the length of every row in the periodic table. And by coupling to the "
    "field of its own orbit, spin sets up the fine splitting of the levels we just "
    "found.",
    difficulty="advanced",
    estimate="160–195 min",
)

## Notebook overview

Movement IV opens by settling an account. Movement I built the *kinematics* of the electron's spin — the
two-state space ([§6.4](stern-gerlach-qubit.ipynb)), the Pauli algebra and uncertainty ([§6.6](pauli-uncertainty.ipynb)), precession and Rabi flopping ([§6.7](time-evolution.ipynb)), the
Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)) — and Movement III sharpened it: [§6.14](angular-momentum-algebra.ipynb) derived spin-$\tfrac12$ as the smallest
representation of the rotation algebra, and [§6.15](orbital-angular-momentum.ipynb) showed that orbital motion *cannot* produce
half-integers. What none of that supplied is the **physics**: what spin *does* in the world. That is this
notebook.

The new content is four things, and we take pains not to re-teach what is already built. First, the
electron's **magnetic moment** and its **anomalous $g$-factor**: a spin produces a magnetic moment about
**twice** as strong as the same amount of *orbital* angular momentum would — $g_s\approx2$ against
$g_l=1$ — a fact Schrödinger's theory cannot explain, which falls out exactly of the Dirac equation
($g=2$) and whose tiny excess $g-2$ is the most precisely tested prediction in all of physics (QED).
Second, the **Zeeman energy** $H=-\boldsymbol\mu\cdot\mathbf B$: a magnetic field gives the two spin
states different energies, splitting them by $g_s\mu_B B$ and precessing a tilted spin at the
$g$-corrected **Larmor frequency** — the working principle of NMR, ESR, MRI, and the spin qubit. This is
where the Stern–Gerlach experiment of [§6.4](stern-gerlach-qubit.ipynb) comes full circle: the beam split because the two spin states
have different energies in a field. Third, the **composite state**: an electron is a spatial orbital
*tensored with* a spin, $|n,l,m_l\rangle\otimes|m_s\rangle$, so each hydrogen shell's $n^2$ orbitals
become $2n^2$ states — the factor of two that [§6.17](hydrogen-atom.ipynb) had to *assume*, now **derived**, and the origin of the
periodic table's row lengths. Fourth, the **spin–orbit coupling** $\propto\mathbf L\cdot\mathbf S$: the
electron's spin couples to the magnetic field of its own orbital motion, splitting each level into
total-angular-momentum sublevels — the atomic **fine structure**. Evaluating $\mathbf L\cdot\mathbf S$
turns out to require knowing how to *add* the angular momenta $\mathbf L$ and $\mathbf S$, which is exactly
the subject of the next notebook: §6.18 poses the question that [§6.19](addition-angular-momenta.ipynb) answers.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the spin matrices $\mathbf S=\tfrac\hbar2\boldsymbol\sigma$
(from [§6.6](pauli-uncertainty.ipynb)/[§6.14](angular-momentum-algebra.ipynb)), the Zeeman Hamiltonian as a matrix, `numpy.kron` for the spatial$\otimes$spin product,
`scipy.linalg.expm` for precession, `numpy.linalg.eigh` for spectra, and $\mathbf L\cdot\mathbf S=
\tfrac12(J^2-L^2-S^2)$ from the [§6.14](angular-momentum-algebra.ipynb) angular-momentum matrices.

> **Conventions and units.** We set $\hbar=1$ and the Bohr magneton $\mu_B=1$, so energies are measured
> in units of $\mu_B$ and the field $B$ in the corresponding units; the $g$-factors are $g_l=1$
> (orbital) and $g_s\approx2.0023$ (spin). The field is along $z$. The spin operators are $\mathbf S=
> \tfrac\hbar2\boldsymbol\sigma$, i.e. the $j=\tfrac12$ matrices of [§6.14](angular-momentum-algebra.ipynb). **Anti-redundancy:** the Pauli
> algebra ([§6.6](pauli-uncertainty.ipynb)), the Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb)), and the mechanics of precession/Rabi ([§6.7](time-evolution.ipynb)) are *used and cited*
> here, not re-derived. See Sakurai & Napolitano and Griffiths (spin, magnetic moments, the Zeeman
> effect, spin–orbit coupling); and Notebooks [§6.4](stern-gerlach-qubit.ipynb) (Stern–Gerlach), [§6.6](pauli-uncertainty.ipynb) (Pauli), [§6.7](time-evolution.ipynb) (precession/Rabi),
> [§6.8](bloch-sphere-entanglement.ipynb) (tensor product/entanglement), [§6.14](angular-momentum-algebra.ipynb) (spin from the algebra), [§6.17](hydrogen-atom.ipynb) (the $2n^2$ asserted).

## Theory in brief

### Spin as intrinsic angular momentum (recap, cited)

The electron carries an intrinsic angular momentum $\mathbf S$ with $s=\tfrac12$, living in a
two-dimensional internal space with $\mathbf S=\tfrac\hbar2\boldsymbol\sigma$ ([§6.6](pauli-uncertainty.ipynb), [§6.14](angular-momentum-algebra.ipynb)). It has **no**
position-space wavefunction — it is not "the electron spinning," but a genuine internal degree of freedom
the rotation algebra demands ([§6.14](angular-momentum-algebra.ipynb)) and orbital motion cannot supply ([§6.15](orbital-angular-momentum.ipynb)). We take this as given and
turn to the new physics.

### The magnetic moment and the $g$-factor

A charged particle with angular momentum has a **magnetic moment**. For orbital angular momentum,

```{math}
:label: eq-g-factor
\boldsymbol\mu_L=-g_l\frac{\mu_B}{\hbar}\mathbf L\ (g_l=1),\qquad \boldsymbol\mu_S=-g_s\frac{\mu_B}{\hbar}\mathbf S\ (g_s\approx2.0023) ,
```

with $\mu_B$ the **Bohr magneton**. The spin's $g_s\approx2$ means a spin produces **twice** the magnetic
moment per unit angular momentum that orbital motion does — an anomaly Schrödinger's theory cannot
explain. It emerges exactly from the **Dirac equation** ($g=2$), and the excess $g-2\approx0.00232$ is a
triumph of **quantum electrodynamics** (both are horizons here).

### The Zeeman Hamiltonian and level splitting

In a field $\mathbf B=B\hat z$, the spin energy is

```{math}
:label: eq-zeeman
H=-\boldsymbol\mu_S\cdot\mathbf B=g_s\frac{\mu_B}{\hbar}B\,S_z=\tfrac12 g_s\mu_B B\,\sigma_z ,
```

with eigenstates spin-up/down split by $\Delta E=g_s\mu_B B$. This **Zeeman** splitting is the energy
behind the two Stern–Gerlach beams ([§6.4](stern-gerlach-qubit.ipynb), full circle): the beam split because the two spin states have
different energies in a field. (The full atomic Zeeman effect adds the orbital moment, $g_l\mathbf L+g_s
\mathbf S$.)

### Larmor precession at the $g$-corrected frequency

A spin tilted from $\mathbf B$ precesses about it ([§6.7](time-evolution.ipynb)) at the **Larmor frequency**

```{math}
:label: eq-larmor-g
\omega_L=\frac{g_s\mu_B B}{\hbar},\qquad \langle S_x\rangle(t)=\tfrac12\cos\omega_L t ,
```

now with the $g$-factor made physical. This is the resonance frequency probed by **NMR**, **ESR/EPR**,
and **MRI**, and the frequency at which a spin qubit is driven. The precession itself is the [§6.7](time-evolution.ipynb) result;
what is new is that $\omega_L$ is a measurable property through $g$.

### The composite state: spatial $\otimes$ spin

The full electron state is the **tensor product** of its spatial and spin parts,

```{math}
:label: eq-spatial-spin
|\Psi\rangle=|n,l,m_l\rangle\otimes|m_s\rangle,\qquad \text{each shell: } n^2\times 2=2n^2\ \text{states} ,
```

labelled by $n,l,m_l,m_s=\pm\tfrac12$ (`numpy.kron`). Spin's factor-two degeneracy, independent of the
spatial state, turns hydrogen's $n^2$ orbitals ([§6.17](hydrogen-atom.ipynb)) into $2n^2$ states — $2,8,18,32$, the factor of two
now **derived**. This is the origin of the periodic table's rows (filled by the Pauli exclusion principle
of [§6.20](identical-particles.ipynb)). A spatial–spin state can be a product or, in general, **entangled** ([§6.8](bloch-sphere-entanglement.ipynb)) — and spin–orbit
coupling entangles them.

### Spin–orbit coupling: the bridge

In the atom's rest frame the electron sees the nucleus orbiting it, a current loop whose magnetic field
couples to the spin — the **spin–orbit** interaction $H_{SO}\propto\mathbf L\cdot\mathbf S$. With
$\mathbf J=\mathbf L+\mathbf S$,

```{math}
:label: eq-spin-orbit
\mathbf L\cdot\mathbf S=\tfrac12\!\left(J^2-L^2-S^2\right),\qquad \langle\mathbf L\cdot\mathbf S\rangle_j=\tfrac{\hbar^2}{2}\big[j(j+1)-l(l+1)-s(s+1)\big] ,
```

so spin–orbit coupling **splits** each orbital level into distinct-$j$ sublevels (an $l=1$ level splits
into $j=\tfrac32$ and $j=\tfrac12$) — the **fine structure** of atomic spectra (the sodium D-line
doublet), computed perturbatively in [§6.21](perturbation-fine-structure.ipynb). But evaluating $\mathbf L\cdot\mathbf S$ requires the
eigenstates of $J^2=(\mathbf L+\mathbf S)^2$ — i.e. how to **add** $\mathbf L$ and $\mathbf S$. That is
exactly the next notebook ([§6.19](addition-angular-momenta.ipynb)).

## Setup

The data are the series palette, the conventions $\hbar=\mu_B=1$ with the two $g$-factors
$g_l=1$ and $g_s\approx2.0023$, the spin-$\tfrac12$ matrices $\mathbf S=\tfrac\hbar2
\boldsymbol\sigma$ every exercise below acts on, and the Larmor frequency $\omega_L=g\mu_B
B/\hbar$ — a one-line conversion from field strength to frequency, an input to the physics
rather than the physics itself. The instrument is the generic construction of
$J_x,J_y,J_z,J^2,J_\pm$ for arbitrary $j$, built from scratch in
[§6.14](angular-momentum-algebra.ipynb) and restated here so both the spin and the orbital
matrices are on hand. The two operators this notebook is *about* are deliberately absent: you
build the Zeeman Hamiltonian in Exercise 2, the spin–orbit operator $\mathbf L\cdot\mathbf S$
in Exercise 6, and the driven-spin evolution in Exercise 7.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import expm

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT  # data: the series palette
RED = "#c1121f"

# data: the conventions — ℏ=1, energies measured in Bohr magnetons, and the two g-factors that
# the whole notebook turns on (the orbital 1 against the electron's anomalous ≈2)
HBAR = 1.0
MU_B = 1.0  # Bohr magneton (energies measured in units of μ_B)
G_L = 1.0  # orbital g-factor
G_S = 2.0023193  # electron spin g-factor (Dirac g=2 plus the QED anomaly)


# built from scratch in §6.14 (Exercise 1, where the J_± matrix elements are assembled rung by
# rung); restated here as an instrument — this notebook spends the matrices, it does not rebuild
# them, and the spin–orbit operator you write in Exercise 6 calls it for both factors.
def angular_momentum_matrices(j, hbar=HBAR):
    r"""The angular-momentum matrices $J_x,J_y,J_z,J^2,J_+,J_-$ for a given $j$ — reused from §6.14.

    Built from the matrix elements $J_\pm|j,m\rangle=\hbar\sqrt{j(j+1)-m(m\pm1)}|j,m\pm1\rangle$ in the
    descending-$m$ basis; $s=\tfrac12$ gives the spin operators $\mathbf S=\tfrac\hbar2\boldsymbol\sigma$.
    """
    dim = int(round(2 * j + 1))
    m = np.arange(j, -j - 1, -1.0)
    Jz = hbar * np.diag(m).astype(complex)
    Jp = np.zeros((dim, dim), dtype=complex)
    Jm = np.zeros((dim, dim), dtype=complex)
    for a in range(dim):
        ma = m[a]
        cp = j * (j + 1) - ma * (ma + 1)
        if cp > 1e-12:
            Jp[a - 1, a] = hbar * np.sqrt(cp)
        cm = j * (j + 1) - ma * (ma - 1)
        if cm > 1e-12:
            Jm[a + 1, a] = hbar * np.sqrt(cm)
    Jx = (Jp + Jm) / 2
    Jy = (Jp - Jm) / 2j
    J2 = Jx @ Jx + Jy @ Jy + Jz @ Jz
    return Jx, Jy, Jz, J2, Jp, Jm


# data: the spin-½ operators S = (ℏ/2)σ — the specimen every exercise acts on (§6.6/§6.14)
SX, SY, SZ, _, _, _ = angular_momentum_matrices(0.5)


# data: the field-to-frequency conversion ω_L = gμ_B B/ℏ — a scalar restatement of the Zeeman gap
# you build in Exercise 2, carrying no construction of its own.
def larmor_frequency(B, g):
    r"""The Larmor precession frequency $\omega_L=g\mu_B B/\hbar$ {eq}`eq-larmor-g`."""
    return g * MU_B * B / HBAR

## Exercise 1 — The magnetic moment and the $g$-factor

Angular momentum and magnetism come together: a charged particle with angular momentum carries a
magnetic moment, $\boldsymbol\mu_L=-g_l(\mu_B/\hbar)\mathbf L$ with $g_l=1$ for orbital motion and
$\boldsymbol\mu_S=-g_s(\mu_B/\hbar)\mathbf S$ with $g_s\approx2.0023$ for spin {eq}`eq-g-factor`.
The honest way to compare the two is the **gyromagnetic ratio** $\gamma=g\mu_B/\hbar$, the moment
produced *per unit angular momentum*, because it divides out how much angular momentum each kind
carries and leaves nothing but the $g$-factor. That $g_s$ is about twice $g_l$ is the electron's
**anomalous magnetic moment** — genuinely new physics, unexplainable in Schrödinger's theory, exact
in the Dirac equation ($g=2$), and in its excess $g-2\approx0.00232$ the most precisely tested
prediction of quantum electrodynamics.

1. Compute the two gyromagnetic ratios $\gamma_S=g_s\mu_B/\hbar$ and $\gamma_L=g_l\mu_B/\hbar$.
2. Compute the ratio $\gamma_S/\gamma_L=g_s/g_l$ and confirm it is $\approx2$ — spin produces twice
   the magnetic moment of the same orbital angular momentum.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    np.array([G_S, G_L]),
    np.array([2.0, 1.0]),
    "the electron spin g-factor is ≈2, twice the orbital g-factor — spin's anomalous magnetic moment",
    atol=1e-2,
)

## Exercise 2 — The Zeeman Hamiltonian and level splitting

Put that moment in a field $\mathbf B=B\hat z$ and it acquires an energy, $H=-\boldsymbol\mu_S
\cdot\mathbf B=g_s(\mu_B/\hbar)B\,S_z=\tfrac12 g_s\mu_B B\,\sigma_z$ {eq}`eq-zeeman` — the
**Zeeman** Hamiltonian, a $2\times2$ matrix that is nothing but the spin operator $S_z$ scaled by
the field. It is already diagonal in the $S_z$ basis, so its eigenvalues are $\pm\tfrac12 g_s
\mu_B B$ and the two spin states are separated by $\Delta E=g_s\mu_B B$: a gap that opens
**linearly** from zero as the field is turned on, with the $g$-factor as its slope. This is the
energy behind the two Stern–Gerlach beams ([§6.4](stern-gerlach-qubit.ipynb), full circle) — the
beam split because these two states cost different energy in a field. (The full atomic Zeeman
effect adds the orbital moment, $g_l\mathbf L+g_s\mathbf S$.)

1. Write `zeeman_hamiltonian(B, g)`, returning $H=g\mu_B B\,S_z/\hbar$ on the spin-$\tfrac12$
   space.
2. Diagonalize it with `numpy.linalg.eigh` at several field strengths.
3. Confirm the two spin states split by $\Delta E=g_s\mu_B B$.
4. Plot the two levels against $B$ and read off the linear splitting.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
B_test = 1.5
E_test = np.linalg.eigvalsh(zeeman_hamiltonian(B_test, G_S))
validate.close(
    E_test[-1] - E_test[0],
    G_S * MU_B * B_test,
    "the Zeeman splitting is g μ_B B",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — Larmor precession at the $g$-corrected frequency

A spin *aligned* with the field simply sits at one of those two energies; a spin **tilted** from it
is a superposition of both, and the phase between them turns — the spin **precesses** about
$\mathbf B$ ([§6.7](time-evolution.ipynb)). The rate is the Zeeman gap divided by $\hbar$: the
**Larmor frequency** $\omega_L=g_s\mu_B B/\hbar$ {eq}`eq-larmor-g`, supplied by the
`larmor_frequency` helper. Prepare the spin along $x$, i.e.
$(|\!\uparrow\rangle+|\!\downarrow\rangle)/\sqrt2$, and the transverse component follows
$\langle S_x\rangle(t)=\tfrac12\cos\omega_L t$ while $\langle S_z\rangle$ — the projection along
$\mathbf B$, fixed by energy conservation — does not move at all. The mechanics are entirely those
of [§6.7](time-evolution.ipynb); what is new is that $\omega_L$ now carries the physical
$g$-factor, which makes it the resonance frequency of NMR, ESR, and MRI, and the frequency at
which a spin qubit is driven.

1. Start a spin along $x$ and evolve it under the `zeeman_hamiltonian` you wrote in Exercise 2,
   using `scipy.linalg.expm` (the time-evolution method of [§6.7](time-evolution.ipynb)).
2. Compute $\langle S_x\rangle(t)$ and confirm it is $\tfrac12\cos\omega_L t$ with
   $\omega_L=g_s\mu_B B/\hbar$.
3. Plot all three components and watch the spin precess rigidly about the field.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    larmor_ok,
    "the spin precesses at the Larmor frequency ω_L = g μ_B B/ℏ, with ⟨S_x⟩(t)=½cos(ω_L t)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The composite state: spatial $\otimes$ spin

An electron is not a wavefunction *or* a spin but both at once, and the two live in different
spaces, so the full state is their **tensor product** $|\Psi\rangle=|n,l,m_l\rangle\otimes|m_s
\rangle$ {eq}`eq-spatial-spin`, assembled numerically with `numpy.kron` and of dimension
$(2l+1)\times2$. Operators inherit the same structure: a spin operator acts on the composite
space as $I_{\text{orb}}\otimes S_z$ and an orbital operator as $L_z\otimes I_{\text{spin}}$,
each leaving the other's factor alone, so the two **commute**. That independence is the whole
point — it is why spin *multiplies* the count of states in the next exercise rather than
rearranging them, and why $m_l$ and $m_s$ can be specified together as two of the electron's four
quantum numbers.

1. Form $|\Psi\rangle=|\text{orbital}\rangle\otimes|\text{spin}\rangle$ with `numpy.kron` from an
   orbital state in the $(2l+1)$-dimensional $l$-space and a spin state $|m_s\rangle$, and check
   its dimension is $(2l+1)\times2$.
2. Build $L_z\otimes I_{\text{spin}}$ and $I_{\text{orb}}\otimes S_z$ and read off $\langle
   L_z\rangle=m_l$ and $\langle S_z\rangle=m_s$ in that state.
3. Confirm the two operators commute.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    tensor_ok,
    "the electron's state is orbital⊗spin (numpy.kron): it carries n,l,m_l,m_s and orbital/spin operators act on independent, commuting factors",
)

## Exercise 5 — The $2n^2$ counting: completing hydrogen

Hydrogen's shell $n$ holds $n^2$ spatial orbitals ([§6.17](hydrogen-atom.ipynb)), which is the sum
$\sum_{l=0}^{n-1}(2l+1)$ over the subshells. Because the spin factor is independent of the spatial
one {eq}`eq-spatial-spin`, every one of those orbitals comes in two copies, $m_s=\pm\tfrac12$, and
the shell holds $2n^2=2,8,18,32$ states. That is the factor of two [§6.17](hydrogen-atom.ipynb) had
to *assume*, now **derived** — and, once the Pauli exclusion principle of
[§6.20](identical-particles.ipynb) forbids two electrons from sharing a state, it is the length of
each row of the periodic table. Spin completes the atom.

1. Count the spatial orbitals per shell as $\sum_{l<n}(2l+1)$ for $n=1,\dots,4$ and confirm the
   total is $n^2$.
2. Multiply by the two spin states and confirm the shell capacities are $2n^2=2,8,18,32$.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.check(
    counts_ok,
    "each hydrogen shell holds 2n² states = 2,8,18,32 — spin supplies the factor of two completing hydrogen's degeneracy",
)

## Exercise 6 — Spin–orbit coupling $\mathbf L\cdot\mathbf S$

In the atom's rest frame the electron sees the nucleus orbiting *it*, a current loop whose magnetic
field couples to its spin: the **spin–orbit** interaction $H_{SO}\propto\mathbf L\cdot\mathbf S$
{eq}`eq-spin-orbit`. On the orbital$\otimes$spin space of Exercise 4 the operator is
$\mathbf L\cdot\mathbf S=L_x\otimes S_x+L_y\otimes S_y+L_z\otimes S_z$ — three `numpy.kron`
products, each pairing an $l$-matrix with a spin-$\tfrac12$ matrix, both from
`angular_momentum_matrices`. Unlike everything before it, this operator does **not** commute with
$L_z$ or $S_z$ separately: it entangles the two factors. Writing $\mathbf J=\mathbf L+\mathbf S$
gives $\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$, so its eigenvalues are
$\tfrac12[j(j+1)-l(l+1)-s(s+1)]$ (with $\hbar=1$), one for each $j=l\pm\tfrac12$, each carrying
$2j+1$ states. An $l=1$ level therefore splits into $j=\tfrac32$ and $j=\tfrac12$ — the **fine
structure** of atomic spectra, the reason the sodium D line is a doublet, computed perturbatively
in [§6.21](perturbation-fine-structure.ipynb). Reading those sublevels off required diagonalizing
$J^2=(\mathbf L+\mathbf S)^2$, i.e. knowing how to *add* $\mathbf L$ and $\mathbf S$ — exactly the
next notebook ([§6.19](addition-angular-momenta.ipynb)).

1. Write `spin_orbit_LS(l, s=0.5)`, assembling $\mathbf L\cdot\mathbf S$ on the
   orbital$\otimes$spin space as the sum of the three `numpy.kron` products of the
   `angular_momentum_matrices` at $j=l$ and $j=s$. **Write this one yourself** — the
   implementation is the lesson.
2. Diagonalize it with `numpy.linalg.eigh` for $l=1$ and $l=2$, and compare the distinct
   eigenvalues to $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$ for $j=l\pm\tfrac12$.
3. Confirm the multiplicities are $2j+1$, so each orbital level splits into exactly two sublevels.
4. Draw the split $l=1$ level and its two $j$ sublevels.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
l_chk = 1
validate.close(
    np.array(sorted(set(np.round(np.linalg.eigvalsh(spin_orbit_LS(l_chk, s)), 6)))),
    np.array(
        sorted(
            0.5 * (j * (j + 1) - l_chk * (l_chk + 1) - s * (s + 1))
            for j in [l_chk + s, l_chk - s]
        )
    ),
    "L·S = ½(J²−L²−S²) splits a level into j-multiplets ½[j(j+1)−l(l+1)−s(s+1)] — the origin of fine structure",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — The electron-spin qubit in a field *(student)*

A static field $B\hat z$ turns one electron spin into a **qubit**: two levels separated by the
Zeeman gap $\hbar\omega_L$ {eq}`eq-zeeman`, {eq}`eq-larmor-g`. To move it between them, add a weak
transverse oscillating field $B_1\cos(\omega_d t)\hat x$, which makes the Hamiltonian
time-dependent, $H(t)=\tfrac12\omega_0\sigma_z+\tfrac12\omega_1\cos(\omega_d t)\sigma_x$. A
time-dependent $H$ has no single matrix exponential, so the evolution is built by **stepping**:
freeze $H$ at the current time, exponentiate it over a short $dt$ (`scipy.linalg.expm`), apply it,
advance. Started from spin-down, the probability of ending spin-up is negligible for almost every
drive frequency, but rises to a sharp peak at $\omega_d=\omega_L$: only there is the drive in step
with the qubit, and only there does it coherently pump Rabi oscillations
([§6.7](time-evolution.ipynb)) between the two states. That peak *is* magnetic resonance — the
line ESR and NMR sweep for, the pulse that executes a gate on a spin qubit, Movement I's
kinematics meeting this notebook's energetics.

1. Write `flip_probability(w_drive)`, evolving a spin from spin-down under $H(t)$ above in small
   steps with `scipy.linalg.expm` and returning the probability of having flipped to spin-up.
   **Write this one yourself** — the implementation is the lesson.
2. Scan the drive frequency $\omega_d$ across the Larmor frequency and record the flip probability
   at each.
3. Show it peaks at $\omega_d=\omega_L$ and is small far off resonance.
4. Plot the resonance line.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    resonance_ok,
    "an electron spin in a field is a qubit addressed at its Larmor frequency: the driven spin flips resonantly, peaking sharply at ω_d=ω_L",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 8 — The electron, completed *(synthesis)*

Movement I taught us how a spin *behaves* — its states, its algebra, its precession, its sphere. This
notebook gave that spin a physical **body**: a magnetic moment twice as strong as motion would grant
($g_s\approx2$, a Dirac/QED fact), an energy in a field that splits and turns it (the Zeeman effect and
Larmor precession, the engine of magnetic resonance), a place in the atom that doubles every shell and
sets the periodic table's rhythm ($2n^2$, the factor of two now derived), and a coupling to its own orbit
that will split the spectral lines ($\mathbf L\cdot\mathbf S$).

There is no new computation here: the completed electron is the result. It is now fully
described — a spatial wavefunction *and* a spin, four quantum numbers, one particle. But the spin–orbit
coupling posed a question we cannot yet answer: to evaluate $\mathbf L\cdot\mathbf S$ we had to
diagonalize $J^2=(\mathbf L+\mathbf S)^2$ — we needed the eigenstates of the *sum* of two angular momenta.
The next notebook ([§6.19](addition-angular-momenta.ipynb)) builds exactly that machinery, the **addition of angular momenta** and the
**Clebsch–Gordan coefficients**, the last tool required before fine structure ([§6.21](perturbation-fine-structure.ipynb)) and the
many-electron atom ([§6.20](identical-particles.ipynb)).

Notice how the number **two** keeps appearing: two spin states, twice the magnetic moment, $2n^2$
electrons per shell, a $g$-factor of $2$. It is the same two each time — the dimension of the smallest
nonzero angular momentum, the $j=\tfrac12$ of [§6.14](angular-momentum-algebra.ipynb) — propagating outward from an algebra into the
magnetism of matter and the length of a row of the periodic table.

## Notebook summary

The physics of spin — the opening of Movement IV.

- **The $g$-factor** {eq}`eq-g-factor`: spin's magnetic moment is $g_s\approx2$ times $\mu_B$ per unit
  angular momentum — twice the orbital $g_l=1$ (Dirac's $g=2$ plus the QED anomaly $g-2$).
- **The Zeeman effect** {eq}`eq-zeeman`: $H=\tfrac12 g_s\mu_B B\,\sigma_z$ splits the spin states by
  $\Delta E=g_s\mu_B B$ — the energy behind the Stern–Gerlach beams ([§6.4](stern-gerlach-qubit.ipynb)).
- **Larmor precession** {eq}`eq-larmor-g`: a tilted spin precesses at $\omega_L=g_s\mu_B B/\hbar$
  (`scipy.linalg.expm`) — the frequency of NMR, ESR, and MRI.
- **Spatial $\otimes$ spin** {eq}`eq-spatial-spin`: the electron state is $|n,l,m_l\rangle\otimes|m_s
  \rangle$ (`numpy.kron`); spin's factor of two makes hydrogen's shells hold $2n^2=2,8,18,32$ states.
- **Spin–orbit coupling** {eq}`eq-spin-orbit`: $\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$ splits a
  level into $j=l\pm\tfrac12$ sublevels — fine structure — and requires adding $\mathbf L$ and $\mathbf S$.

The electron now has a body and a place in the atom. What remains is to learn to add its two angular
momenta — the subject of the next notebook.

## Outlook

- **The addition of angular momenta and Clebsch–Gordan coefficients ([§6.19](addition-angular-momenta.ipynb))**: the eigenstates of
  $\mathbf J=\mathbf L+\mathbf S$, needed to evaluate $\mathbf L\cdot\mathbf S$.
- **Identical particles and the Pauli exclusion principle ([§6.20](identical-particles.ipynb))**: filling the $2n^2$ states to build
  atoms and the periodic table.
- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))**: the spin–orbit and relativistic corrections by perturbation theory; the
  anomalous Zeeman effect.
- **The electron's $g-2$ and QED; the Dirac equation's $g=2$** (horizons — the deep origin of the
  anomalous moment).
- **Cross-reference** [§6.4](stern-gerlach-qubit.ipynb) (Stern–Gerlach), [§6.6](pauli-uncertainty.ipynb) (Pauli), [§6.7](time-evolution.ipynb) (precession/Rabi), [§6.8](bloch-sphere-entanglement.ipynb) (tensor
  product/entanglement), [§6.14](angular-momentum-algebra.ipynb) (spin from the algebra), [§6.17](hydrogen-atom.ipynb) (the $2n^2$), and forward to [§6.19](addition-angular-momenta.ipynb), [§6.20](identical-particles.ipynb), [§6.21](perturbation-fine-structure.ipynb).

In [ ]:
from ecp.style import footer

footer()